In [ ]:
# tfvggt: training-free faster VGGT. One cell, runs unchanged on Kaggle and on a local machine.
# Kaggle: Settings -> Accelerator "GPU T4 x1", Internet "On". Results: /kaggle/working/results (local: ./results).
# Change ARGS only; `tfvggt run --help` and `tfvggt configs` list every option and configuration.
ARGS = "--dataset tum:fr1_xyz --frames 50 100 --configs recommended"
REPO = "https://github.com/<user>/tfvggt.git"  # used only when no local copy of the code is found

import glob, os, shutil, subprocess, sys
from pathlib import Path

try:  # plain Python on Windows: do not fail on non-ASCII paths (Jupyter/Kaggle output is UTF-8 already)
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass


def sh(cmd):
    """Run a command and stream its output into the cell."""
    env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env,
                         text=True, encoding="utf-8", errors="replace", bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")


def find_source():
    for d in (Path.cwd(), *Path.cwd().parents):  # 1. this notebook inside a checkout (<repo>/notebooks/)
        if (d / "src" / "tfvggt" / "cli.py").is_file():
            return d
    for p in glob.glob("/kaggle/input/**/src/tfvggt/cli.py", recursive=True):  # 2. code attached as a Kaggle dataset
        return Path(p).parents[2]
    dest = Path("/tmp/tfvggt-src") if os.path.isdir("/kaggle") else Path.home() / "tfvggt-src"  # 3. GitHub
    if not dest.exists():
        sh(f'git clone -q --depth 1 {REPO} "{dest}"')
    return dest


src = find_source()
if str(src).startswith("/kaggle/input"):  # read-only there: install from a writable copy
    shutil.copytree(src, "/tmp/tfvggt-src", dirs_exist_ok=True)
    src = Path("/tmp/tfvggt-src")
if not os.path.isdir("/kaggle"):
    os.chdir(src)  # local: data/ and results/ go to the root of the checkout
print("tfvggt source:", src, "| working directory:", os.getcwd())
sh(f'"{sys.executable}" -m pip install -q "{src}"')  # plain install: works for any path (no .pth file)
sh(f'"{sys.executable}" -m tfvggt run {ARGS}')
